Imports and Configs

In [ ]:
import math
import warnings
from dataclasses import dataclass
from enum import Enum
from typing import Any, Dict, Iterable, List, Optional, Tuple
import ijson
import pandas as pd
import optuna
import pandas as pd
from textblob import TextBlob
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.neighbors import NearestNeighbors, BallTree
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore", category=UserWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

@dataclass()
class config:
    S3_BUCKET : str = "***"
    S3_PREFIX : str = "***"
    ACCOUNTS_PATH : str = ""
    CALL_PATH : str = ""
    STOCK_PATH : str = ""
    RANDOM_STATE : int = 42
    SCORING_FREQ : str = "W-MON"
    DISPENSER_TARGET_DAYS : int = 7
    LOOKBACK_DAYS : int = 180
    GEO_RADIUS_KM : float = 10.0
    STOCK_THRESHOLD : float = 50.0
    COLD_MIN_VISITS : int = 3
    EPSILON_EXPLORE : float = 0.10
    VACCINES_BU : str = "VACCINES DOMESTIC"
    VACCINES_ITEMS_KEY: str = "orderLineItems"
    NUTRITION_ITEMS_KEY: str = "nutritionOrderLineItems"
    POB_POSITIVE_STATUSES: Tuple[str, ...] = ("delivered", "partially fulfilled")
    POB_CANCELLED_STATUS: str = "cancelled"
    CATEGORICAL_FEATURES: Tuple[str, ...] = (
        "account_type","state","region","zone","pool_name","territory_name",
        "town_type","specialty","practice_type","account_potential",
        "scheme_name_last_30d","call_mode_last","call_type_last","time_of_day_last",
    )

CFG = Config(
    ACCOUNTS_PATH=f"***",
    CALLS_PATH=f"***",
    STOCK_PATH=f"***",
)

class MongoKeys(str, Enum):
    OID = "$oid"
    DATE = "$date"
    NUMBER_LONG = "$numberLong"

class AccountType(str, Enum):
    DOCTOR = "Doctor"
    RETAILER = "Retailer"
    INSTITUTION = "Institution"
    DISTRIBUTOR = "Distributor"
    UNKNOWN = "UNKNOWN"

Utilities

In [ ]:
class DataUtils:
    @staticmethod
    def extract_oid(field: Any) -> Optional[str]:
        if isinstance(field, dict) and MongoKeys.OID.value in field:
            return field[MongoKeys.OID.value]
        return None if field is None else str(field)

    @staticmethod
    def extract_date(field: Any) -> pd.Timestamp:
        if not field:
            return pd.NaT
        if isinstance(field, dict) and MongoKeys.DATE.value in field:
            val = field[MongoKeys.DATE.value]
            if isinstance(val, dict) and MongoKeys.NUMBER_LONG.value in val:
                return pd.to_datetime(int(val[MongoKeys.NUMBER_LONG.value]), unit="ms", utc=True, errors="coerce")
            return pd.to_datetime(val, utc=True, errors="coerce")
        return pd.to_datetime(field, utc=True, errors="coerce")

    @staticmethod
    def stream_json_file(filepath: str, limit: Optional[int] = None):
        if filepath.startswith("s3://"):
            bucket, key = filepath[5:].split("/", 1)
            s3 = boto3.client("s3")
            response = s3.get_object(Bucket=bucket, Key=key)
            stream = response["Body"]
        else:
            stream = open(filepath, "rb")
        try:
            for i, item in enumerate(ijson.items(stream, "item")):
                if limit is not None and i >= limit:
                    break
                yield item
        finally:
            stream.close()

    @staticmethod
    def to_rad(lon: pd.Series, lat: pd.Series):
        return np.column_stack([np.radians(lat.astype(float).values), np.radians(lon.astype(float).values)])

Parsers

In [ ]:
class AccountsParser:
    @staticmethod
    def _std_type(raw):
        if not raw:
            return AccountType.UNKNOWN.value
        raw = str(raw).strip()
        if raw in {"Hospital", "NursingHome", "MaternityHome", "Clinic", "Institution"}:
            return AccountType.INSTITUTION.value
        if raw in {"Distributor", "Stockist"}:
            return AccountType.DISTRIBUTOR.value
        if raw in {"Doctor", "Retailer"}:
            return raw
        return raw

    def parse(self, stream):
        rows = []
        for doc in stream:
            locations = doc.get("locations") or []
            effective_active = 0
            pool_name = "UNKNOWN_POOL"
            territory_name = "UNKNOWN_TERRITORY"
            region = "UNKNOWN_REGION"
            zone = "UNKNOWN_ZONE"
            town_type = "UNKNOWN_TOWN"
            business_units = []

            for loc in locations:
                if isinstance(loc, dict) and str(loc.get("status", "")).lower() == "active":
                    effective_active = 1
                    pool_name = loc.get("poolName", pool_name)
                    territory_name = loc.get("name", territory_name)
                    business_units.extend(loc.get("businessUnits", []) or [])
                    h = loc.get("locationHierarchy", {}) or {}
                    region = h.get("Region", region)
                    zone = h.get("Zone", zone)

            if territory_name == "UNKNOWN_TERRITORY":
                location_dict = doc.get("location") or {}
                if isinstance(location_dict, dict):
                    for _, loc_data in location_dict.items():
                        if isinstance(loc_data, dict):
                            territory_name = loc_data.get("name", territory_name)
                            pool_name = loc_data.get("poolName", pool_name)
                            town_type = loc_data.get("townType", town_type)
                            h = loc_data.get("locationHierarchy", {}) or {}
                            region = h.get("Region", region)
                            zone = h.get("Zone", zone)
                            break

            geo = doc.get("geoLocation") or {}
            coords = geo.get("coordinates", [None, None])
            lon = float(coords[0]) if len(coords) > 0 and coords[0] is not None else np.nan
            lat = float(coords[1]) if len(coords) > 1 and coords[1] is not None else np.nan

            info = doc.get("information") or {}
            qc = doc.get("qualityCheck") or {}
            addr = doc.get("address") or {}

            account_type = self._std_type(doc.get("accountType"))
            patient_load = 0.0
            account_potential = "UNKNOWN"
            specialty = "UNKNOWN"
            practice_type = "UNKNOWN"

            if account_type == AccountType.DOCTOR.value:
                patient_load = float((info.get("newBornBabiesPerMonth") or 0) + (info.get("infantsSeenPerDayAgeGroup6mTo2y") or 0))
                account_potential = info.get("potential", "UNKNOWN") or "UNKNOWN"
                specialty = info.get("specialty", "UNKNOWN") or "UNKNOWN"
                practice_type = info.get("practiceType", "UNKNOWN") or "UNKNOWN"
            elif account_type in {AccountType.RETAILER.value, AccountType.INSTITUTION.value}:
                patient_load = float(info.get("monthlyNutritionProductsSold") or 0)
                account_potential = info.get("nutritionPotentialClassification", "UNKNOWN") or "UNKNOWN"

            rows.append({
                "account_id": DataUtils.extract_oid(doc.get("_id")),
                "account_name": doc.get("name") if not isinstance(doc.get("name"), dict) else "Unknown",
                "account_type": account_type,
                "effective_active_flag": effective_active,
                "erp_code": doc.get("erpCode"),
                "has_erp_code": int(bool(doc.get("erpCode"))),
                "state": addr.get("state") or "UNKNOWN",
                "city": addr.get("city") or "UNKNOWN",
                "region": region,
                "zone": zone,
                "pool_name": pool_name,
                "territory_name": territory_name,
                "town_type": town_type,
                "business_units": "|".join(sorted(set([str(x) for x in business_units]))),
                "longitude": lon,
                "latitude": lat,
                "profile_quality_score": float(qc.get("Score") or 0.0),
                "specialty": specialty,
                "practice_type": practice_type,
                "patient_load": patient_load,
                "account_potential": account_potential,
                "account_added_on": DataUtils.extract_date(doc.get("addedOn")),
            })
        df = pd.DataFrame(rows)
        if not df.empty:
            df["account_added_on"] = pd.to_datetime(df["account_added_on"], utc=True, errors="coerce")
        return df


class CallsParser:
    @staticmethod
    def _resolve_line_items(doc):
        account = doc.get("account") or {}
        location = account.get("location") or {}
        bu_list = location.get("businessUnits", []) if isinstance(location, dict) else []
        bu_set = {str(x).upper() for x in bu_list}
        if CFG.VACCINES_BU in bu_set:
            key = CFG.VACCINES_ITEMS_KEY
        elif any(x in bu_set for x in CFG.NUTRITION_BUS):
            key = CFG.NUTRITION_ITEMS_KEY
        else:
            key = CFG.NUTRITION_ITEMS_KEY if doc.get(CFG.NUTRITION_ITEMS_KEY) else CFG.VACCINES_ITEMS_KEY
        return doc.get(key) or [], key

    def parse(self, stream):
        headers, lines = [], []
        for doc in stream:
            call_id = DataUtils.extract_oid(doc.get("_id"))
            call_date = DataUtils.extract_date(doc.get("date"))
            acc = doc.get("account") or {}
            account_id = DataUtils.extract_oid(acc.get("accountId"))
            account_type = acc.get("accountType") or "UNKNOWN"

            campaigns = doc.get("campaigns") or []
            promo_inputs = doc.get("marketingInputsUtilized") or []
            remarks = doc.get("conversationRemarks") or ""
            sentiment = TextBlob(remarks).sentiment.polarity if remarks else 0.0
            team = doc.get("accompanyingTeamMembers") or []
            pob_status = str(doc.get("pobStatus") or "").strip().lower()
            geo = doc.get("geoLocation") or {}
            coords = geo.get("coordinates", [None, None])

            headers.append({
                "call_id": call_id,
                "account_id": account_id,
                "account_type_in_call": account_type,
                "call_date": call_date,
                "call_mode": doc.get("callMode") or "UNKNOWN",
                "call_type": doc.get("callType") or "UNKNOWN",
                "call_time_of_day": doc.get("time") or "UNKNOWN",
                "scheme_name": (doc.get("scheme") or "").strip() or "NO_SCHEME",
                "campaign_count": len(campaigns),
                "promo_input_count": len(promo_inputs),
                "remarks_sentiment": float(sentiment),
                "manager_present_flag": int(len(team) > 0),
                "distance_from_account": float(doc.get("distanceFromAccountLocation") or 0.0),
                "pob_status": pob_status,
                "call_longitude": float(coords[0]) if len(coords) > 0 and coords[0] is not None else np.nan,
                "call_latitude": float(coords[1]) if len(coords) > 1 and coords[1] is not None else np.nan,
            })

            line_items, items_key = self._resolve_line_items(doc)
            for item in line_items:
                fulfillment = item.get("fulfillmentAccount") or {}
                lines.append({
                    "call_id": call_id,
                    "account_id": account_id,
                    "call_date": call_date,
                    "account_type_in_call": account_type,
                    "sku_code": item.get("skuCode"),
                    "pob_quantity": float(item.get("quantity") or 0.0),
                    "quantity_cancelled": float(item.get("quantityCancelled") or 0.0),
                    "fulfillment_account_id": DataUtils.extract_oid(fulfillment.get("accountId")),
                    "items_key_used": items_key,
                })

        h = pd.DataFrame(headers)
        l = pd.DataFrame(lines)
        for df in (h, l):
            if not df.empty and "call_date" in df.columns:
                df["call_date"] = pd.to_datetime(df["call_date"], utc=True, errors="coerce")
        return h, l


class StockParser:
    def parse(self, stream):
        rows = []
        for doc in stream:
            account = doc.get("account") or {}
            location = account.get("location") or {}
            pool_name = location.get("poolName", "UNKNOWN_POOL") if isinstance(location, dict) else "UNKNOWN_POOL"
            for item in doc.get("inventoryBook") or []:
                rows.append({
                    "distributor_id": DataUtils.extract_oid(account.get("accountId")),
                    "distributor_erp_code": account.get("erpCode"),
                    "pool_name": pool_name,
                    "statement_date": DataUtils.extract_date(doc.get("statementDate")),
                    "uploaded_on": DataUtils.extract_date(doc.get("uploadedOn")),
                    "status": doc.get("status") or "UNKNOWN",
                    "sku_code": item.get("skuCode"),
                    "opening_stock": float(item.get("opening") or 0.0),
                    "purchase_qty": float(item.get("purchase") or 0.0),
                    "secondary_sales_qty": float(item.get("sales") or 0.0),
                    "in_transit": float(item.get("inTransit") or 0.0),
                    "closing_stock": float(item.get("closing") or 0.0),
                })
        df = pd.DataFrame(rows)
        if not df.empty:
            df["statement_date"] = pd.to_datetime(df["statement_date"], utc=True, errors="coerce")
            df["uploaded_on"] = pd.to_datetime(df["uploaded_on"], utc=True, errors="coerce")
        return df

Normalize Raw Tables

In [ ]:
accounts_df = AccountsParser().parse(DataUtils.stream_json_file(CFG.ACCOUNTS_PATH))
calls_h, calls_l = CallsParser().parse(DataUtils.stream_json_file(CFG.CALLS_PATH))
stock_df = StockParser().parse(DataUtils.stream_json_file(CFG.STOCK_PATH))

print(accounts_df.shape, calls_h.shape, calls_l.shape, stock_df.shape)
accounts_df.head()

Entity Resolution

In [ ]:
calls_df = calls_l.merge(
    calls_h,
    on=["call_id", "account_id", "call_date", "account_type_in_call"],
    how="left"
).merge(
    accounts_df,
    on="account_id",
    how="left",
    suffixes=("", "_acct")
)

calls_df["account_type"] = calls_df["account_type"].fillna(calls_df["account_type_in_call"]).fillna("UNKNOWN")

valid = calls_df[
    calls_df["fulfillment_account_id"].notna() &
    calls_df["sku_code"].notna() &
    calls_df["pob_status"].isin(CFG.POB_POSITIVE_STATUSES)
]
fulfillment_map = (
    valid.groupby(["account_id", "sku_code", "fulfillment_account_id"])
    .size().reset_index(name="cnt")
    .sort_values(["account_id", "sku_code", "cnt"], ascending=[True, True, False])
    .groupby(["account_id", "sku_code"]).head(1)
    .rename(columns={"fulfillment_account_id": "mapped_distributor_id"})[
        ["account_id", "sku_code", "mapped_distributor_id"]
    ]
)

print(calls_df.shape, fulfillment_map.shape)

Candidate Generation

In [ ]:
def build_training_ranges(calls_df, stock_df):
    min_call = calls_df["call_date"].min()
    max_call = calls_df["call_date"].max()
    min_stock = stock_df["statement_date"].min() if not stock_df.empty else min_call
    start = max(min_call, min_stock) + pd.Timedelta(days=CFG.LOOKBACK_DAYS)
    end = max_call - pd.Timedelta(days=CFG.DISPENSER_TARGET_DAYS)
    return start, end

def top_skus_by_pool(calls_df, top_n=20):
    x = (
        calls_df[calls_df["sku_code"].notna()]
        .groupby(["pool_name", "sku_code"]).size()
        .reset_index(name="cnt")
        .sort_values(["pool_name", "cnt"], ascending=[True, False])
    )
    x["rk"] = x.groupby("pool_name").cumcount() + 1
    return x[x["rk"] <= top_n][["pool_name", "sku_code"]]

def recent_account_skus(calls_df, scoring_date):
    lo = scoring_date - pd.Timedelta(days=CFG.LOOKBACK_DAYS)
    hist = calls_df[(calls_df["call_date"] < scoring_date) & (calls_df["call_date"] >= lo) & calls_df["sku_code"].notna()]
    return hist[["account_id", "sku_code"]].drop_duplicates()

def stocked_pool_skus(stock_df, scoring_date, top_n=20):
    hist = stock_df[stock_df["statement_date"] <= scoring_date]
    if hist.empty:
        return pd.DataFrame(columns=["pool_name", "sku_code"])
    latest = hist.sort_values(["pool_name", "sku_code", "statement_date", "uploaded_on"]).groupby(["pool_name", "sku_code"]).tail(1)
    latest = latest[latest["closing_stock"] > 0]
    agg = latest.groupby(["pool_name", "sku_code"])["closing_stock"].sum().reset_index()
    agg = agg.sort_values(["pool_name", "closing_stock"], ascending=[True, False])
    agg["rk"] = agg.groupby("pool_name").cumcount() + 1
    return agg[agg["rk"] <= top_n][["pool_name", "sku_code"]]

start_date, end_date = build_training_ranges(calls_df, stock_df)
scoring_dates = pd.date_range(start_date.normalize(), end_date.normalize(), freq=CFG.SCORING_FREQ, tz="UTC")

active = accounts_df[
    (accounts_df["effective_active_flag"] == 1) &
    (accounts_df["account_type"].isin([AccountType.RETAILER.value, AccountType.INSTITUTION.value]))
].copy()

base_cols = [
    "account_id","account_name","account_type","pool_name","territory_name","region","zone",
    "state","city","town_type","specialty","practice_type","patient_load",
    "account_potential","profile_quality_score","longitude","latitude",
    "account_added_on","has_erp_code","effective_active_flag"
]

pool_top = top_skus_by_pool(calls_df, 20)

snapshots = []
for t in scoring_dates:
    base = active[base_cols].copy()
    base["scoring_date"] = t
    cand1 = base.merge(recent_account_skus(calls_df, t), on="account_id", how="inner")
    cand2 = base.merge(pool_top, on="pool_name", how="inner")
    cand3 = base.merge(stocked_pool_skus(stock_df, t, 20), on="pool_name", how="inner")
    tmp = pd.concat([cand1, cand2, cand3], ignore_index=True)
    if not tmp.empty:
        snapshots.append(tmp.drop_duplicates(["account_id", "sku_code", "scoring_date"]))

dispenser_snapshots = pd.concat(snapshots, ignore_index=True) if snapshots else pd.DataFrame()
print(dispenser_snapshots.shape)

7 Day Target

In [ ]:
pos = calls_df[
    calls_df["pob_status"].isin(CFG.POB_POSITIVE_STATUSES) &
    calls_df["sku_code"].notna()
][["account_id", "sku_code", "call_date"]].drop_duplicates()

train_snap = dispenser_snapshots.copy().reset_index(drop=True)
train_snap["snapshot_row"] = np.arange(len(train_snap))

merged = train_snap.merge(pos, on=["account_id", "sku_code"], how="left")
merged["within_window"] = (
    (merged["call_date"] > merged["scoring_date"]) &
    (merged["call_date"] <= merged["scoring_date"] + pd.Timedelta(days=CFG.DISPENSER_TARGET_DAYS))
).astype(int)

y = merged.groupby("snapshot_row")["within_window"].max().rename("y_target")
train_snap = train_snap.merge(y, on="snapshot_row", how="left")
train_snap["y_target"] = train_snap["y_target"].fillna(0).astype(int)
train_snap = train_snap.drop(columns=["snapshot_row"])

train_snap["y_target"].mean()

Historical Feature Engineering

In [ ]:
def add_history_features(snapshot_df, calls_df):
    out = snapshot_df.copy()
    feats = []
    calls_df = calls_df.sort_values("call_date")

    for idx, row in out.iterrows():
        acc = row["account_id"]
        sku = row["sku_code"]
        t = row["scoring_date"]
        lo7 = t - pd.Timedelta(days=7)
        lo30 = t - pd.Timedelta(days=30)
        lo90 = t - pd.Timedelta(days=90)

        hist = calls_df[(calls_df["account_id"] == acc) & (calls_df["call_date"] < t)]
        hist_sku = hist[hist["sku_code"] == sku]
        h7 = hist[hist["call_date"] >= lo7]
        h30 = hist[hist["call_date"] >= lo30]
        h90 = hist[hist["call_date"] >= lo90]
        hsku30 = hist_sku[hist_sku["call_date"] >= lo30]

        pos = hist[hist["pob_status"].isin(CFG.POB_POSITIVE_STATUSES)]
        pos_sku = hist_sku[hist_sku["pob_status"].isin(CFG.POB_POSITIVE_STATUSES)]
        canc = hist[hist["pob_status"] == CFG.POB_CANCELLED_STATUS]

        last_visit = hist["call_date"].max() if not hist.empty else pd.NaT
        last_pos = pos["call_date"].max() if not pos.empty else pd.NaT
        last30 = h30.sort_values("call_date")

        last_scheme = last30["scheme_name"].dropna().iloc[-1] if not last30.empty else "NO_SCHEME"
        last_mode = last30["call_mode"].dropna().iloc[-1] if not last30.empty else "UNKNOWN"
        last_type = last30["call_type"].dropna().iloc[-1] if not last30.empty else "UNKNOWN"
        last_tod = last30["call_time_of_day"].dropna().iloc[-1] if not last30.empty else "UNKNOWN"

        scheme_hist = hist[hist["scheme_name"] == last_scheme] if not hist.empty else hist
        scheme_pos = scheme_hist[scheme_hist["pob_status"].isin(CFG.POB_POSITIVE_STATUSES)] if not scheme_hist.empty else scheme_hist

        feats.append({
            "snapshot_idx": idx,
            "historical_visits": float(len(hist)),
            "visits_last_7d": float(len(h7)),
            "visits_last_30d": float(len(h30)),
            "visits_last_90d": float(len(h90)),
            "days_since_last_visit": float((t - last_visit).days) if pd.notna(last_visit) else 999.0,
            "historical_positive_pobs": float(len(pos)),
            "historical_conversion_rate": float(len(pos) / len(hist)) if len(hist) else 0.0,
            "days_since_last_positive_pob": float((t - last_pos).days) if pd.notna(last_pos) else 999.0,
            "avg_prior_pob_quantity": float(pos["pob_quantity"].mean()) if not pos.empty else 0.0,
            "same_sku_pitches_last_30d": float(len(hsku30)),
            "same_sku_positive_pobs": float(len(pos_sku)),
            "same_sku_conversion_rate": float(len(pos_sku) / len(hist_sku)) if len(hist_sku) else 0.0,
            "unique_skus_pitched_last_90d": float(h90["sku_code"].nunique()),
            "cancellation_count": float(len(canc)),
            "cancellation_ratio": float(len(canc) / len(hist)) if len(hist) else 0.0,
            "total_cancelled_units": float(hist["quantity_cancelled"].sum()) if not hist.empty else 0.0,
            "scheme_calls_last_30d": float(len(last30[last30["scheme_name"] != "NO_SCHEME"])),
            "scheme_positive_last_30d": float(len(last30[(last30["scheme_name"] != "NO_SCHEME") & (last30["pob_status"].isin(CFG.POB_POSITIVE_STATUSES))])),
            "scheme_sensitivity_index": float(len(scheme_pos) / len(scheme_hist)) if len(scheme_hist) else 0.0,
            "campaign_count_last_30d": float(last30["campaign_count"].sum()) if not last30.empty else 0.0,
            "promo_input_count_last_30d": float(last30["promo_input_count"].sum()) if not last30.empty else 0.0,
            "remarks_sentiment_avg_last_30d": float(last30["remarks_sentiment"].mean()) if not last30.empty else 0.0,
            "scheme_name_last_30d": str(last_scheme),
            "call_mode_last": str(last_mode),
            "call_type_last": str(last_type),
            "time_of_day_last": str(last_tod),
        })

    feat_df = pd.DataFrame(feats).set_index("snapshot_idx")
    return out.join(feat_df)

train_df = add_history_features(train_snap, calls_df)
train_df.shape

Supply + Geo + Temporal + Cold-start

In [ ]:
# supply
train_df["snapshot_row"] = np.arange(len(train_df))
train_df = train_df.merge(fulfillment_map, on=["account_id", "sku_code"], how="left")

exact = train_df[train_df["mapped_distributor_id"].notna()][["snapshot_row", "mapped_distributor_id", "sku_code", "scoring_date"]]
if not exact.empty:
    tmp = exact.merge(stock_df.rename(columns={"distributor_id": "mapped_distributor_id"}), on=["mapped_distributor_id", "sku_code"], how="left")
    tmp = tmp[tmp["statement_date"] <= tmp["scoring_date"]]
    if not tmp.empty:
        tmp = tmp.sort_values(["snapshot_row", "statement_date", "uploaded_on"]).groupby("snapshot_row").tail(1)
        tmp = tmp[["snapshot_row", "opening_stock", "purchase_qty", "secondary_sales_qty", "in_transit", "closing_stock"]]
        tmp = tmp.rename(columns={
            "opening_stock": "distributor_opening_stock",
            "purchase_qty": "distributor_purchase_qty",
            "secondary_sales_qty": "distributor_secondary_sales_qty",
            "in_transit": "distributor_in_transit",
            "closing_stock": "distributor_closing_stock",
        })
        train_df = train_df.merge(tmp, on="snapshot_row", how="left")

tmp2 = train_df[["snapshot_row", "pool_name", "sku_code", "scoring_date"]].merge(stock_df, on=["pool_name", "sku_code"], how="left")
tmp2 = tmp2[tmp2["statement_date"] <= tmp2["scoring_date"]]
if not tmp2.empty:
    tmp2 = tmp2.sort_values(["snapshot_row", "statement_date", "uploaded_on"]).groupby("snapshot_row").tail(1)
    tmp2 = tmp2[["snapshot_row", "opening_stock", "purchase_qty", "secondary_sales_qty", "in_transit", "closing_stock"]]
    tmp2 = tmp2.rename(columns={
        "opening_stock": "pool_opening_stock",
        "purchase_qty": "pool_purchase_qty",
        "secondary_sales_qty": "pool_secondary_sales_qty",
        "in_transit": "pool_in_transit",
        "closing_stock": "pool_closing_stock",
    })
    train_df = train_df.merge(tmp2, on="snapshot_row", how="left")

for a, b, c in [
    ("latest_opening_stock", "distributor_opening_stock", "pool_opening_stock"),
    ("latest_purchase_qty", "distributor_purchase_qty", "pool_purchase_qty"),
    ("latest_secondary_sales_qty", "distributor_secondary_sales_qty", "pool_secondary_sales_qty"),
    ("latest_in_transit", "distributor_in_transit", "pool_in_transit"),
    ("latest_closing_stock", "distributor_closing_stock", "pool_closing_stock"),
]:
    train_df[a] = train_df.get(b, np.nan)
    train_df[a] = train_df[a].fillna(train_df.get(c, np.nan))

total_avail = train_df["latest_opening_stock"].fillna(0) + train_df["latest_purchase_qty"].fillna(0) + train_df["latest_in_transit"].fillna(0)
train_df["sell_through_rate"] = np.where(total_avail > 0, train_df["latest_secondary_sales_qty"].fillna(0) / total_avail, 0.0)
avg_daily = train_df["latest_secondary_sales_qty"].fillna(0) / 30.0
train_df["stock_cover_days"] = np.where(avg_daily > 0, train_df["latest_closing_stock"].fillna(0) / avg_daily, 999.0)
train_df["zero_stock_flag"] = (train_df["latest_closing_stock"].fillna(0) <= 0).astype(int)
train_df["low_stock_flag"] = ((train_df["latest_closing_stock"].fillna(0) > 0) & (train_df["latest_closing_stock"].fillna(0) < CFG.STOCK_THRESHOLD)).astype(int)

# geo
disp = accounts_df[
    (accounts_df["effective_active_flag"] == 1) &
    (accounts_df["account_type"].isin([AccountType.RETAILER.value, AccountType.INSTITUTION.value])) &
    accounts_df["longitude"].notna() & accounts_df["latitude"].notna()
]
tgt = train_df[train_df["longitude"].notna() & train_df["latitude"].notna()]
if not disp.empty and not tgt.empty:
    tree = BallTree(DataUtils.to_rad(disp["longitude"], disp["latitude"]), metric="haversine")
    counts = tree.query_radius(DataUtils.to_rad(tgt["longitude"], tgt["latitude"]), r=CFG.GEO_RADIUS_KM / 6371.0, count_only=True)
    train_df["active_pharmacy_density"] = 0
    train_df.loc[tgt.index, "active_pharmacy_density"] = counts
else:
    train_df["active_pharmacy_density"] = 0

# temporal/composite
train_df["account_age_days"] = (train_df["scoring_date"] - train_df["account_added_on"]).dt.days.fillna(999).astype(float)
train_df["is_month_end"] = (train_df["scoring_date"].dt.day >= 25).astype(int)
train_df["month"] = train_df["scoring_date"].dt.month.astype(int)
train_df["quarter"] = train_df["scoring_date"].dt.quarter.astype(int)
train_df["weekofyear"] = train_df["scoring_date"].dt.isocalendar().week.astype(int)

train_df["patient_load_to_pob_ratio"] = train_df["patient_load"].fillna(0) / (train_df["historical_positive_pobs"].fillna(0) + 1)
train_df["effort_to_conversion_ratio"] = train_df["historical_visits"].fillna(0) / (train_df["historical_positive_pobs"].fillna(0) + 1)
train_df["stock_x_same_sku_conv"] = train_df["latest_closing_stock"].fillna(0) * train_df["same_sku_conversion_rate"].fillna(0)
train_df["stock_x_hist_conv"] = train_df["latest_closing_stock"].fillna(0) * train_df["historical_conversion_rate"].fillna(0)

# cold start
train_df["cold_start_flag"] = (train_df["historical_visits"].fillna(0) < CFG.COLD_MIN_VISITS).astype(int)
train_df["cold_start_neighbor_conv_rate"] = 0.0
train_df["cold_start_similarity_score"] = 0.0

train_df.shape

Training Model

In [ ]:
FEATURE_COLUMNS = [
    "account_type", "state", "region", "zone", "pool_name", "territory_name", "town_type",
    "specialty", "practice_type", "account_potential",
    "patient_load", "profile_quality_score", "has_erp_code", "effective_active_flag",
    "historical_visits", "visits_last_7d", "visits_last_30d", "visits_last_90d",
    "days_since_last_visit", "historical_positive_pobs", "historical_conversion_rate",
    "days_since_last_positive_pob", "avg_prior_pob_quantity",
    "same_sku_pitches_last_30d", "same_sku_positive_pobs", "same_sku_conversion_rate",
    "unique_skus_pitched_last_90d", "cancellation_count", "cancellation_ratio", "total_cancelled_units",
    "scheme_calls_last_30d", "scheme_positive_last_30d", "scheme_sensitivity_index",
    "campaign_count_last_30d", "promo_input_count_last_30d", "remarks_sentiment_avg_last_30d",
    "scheme_name_last_30d", "call_mode_last", "call_type_last", "time_of_day_last",
    "latest_opening_stock", "latest_purchase_qty", "latest_secondary_sales_qty", "latest_in_transit",
    "latest_closing_stock", "sell_through_rate", "stock_cover_days", "zero_stock_flag", "low_stock_flag",
    "active_pharmacy_density",
    "account_age_days", "is_month_end", "month", "quarter", "weekofyear",
    "patient_load_to_pob_ratio", "effort_to_conversion_ratio", "stock_x_same_sku_conv", "stock_x_hist_conv",
    "cold_start_flag", "cold_start_neighbor_conv_rate", "cold_start_similarity_score",
]

cat_cols = [c for c in CFG.CATEGORICAL_FEATURES if c in FEATURE_COLUMNS]

for c in FEATURE_COLUMNS:
    if c not in train_df.columns:
        train_df[c] = "UNKNOWN" if c in cat_cols else 0.0

for c in cat_cols:
    train_df[c] = train_df[c].fillna("UNKNOWN").astype(str)

for c in [x for x in FEATURE_COLUMNS if x not in cat_cols]:
    train_df[c] = pd.to_numeric(train_df[c], errors="coerce").fillna(0.0)

train_df = train_df.sort_values("scoring_date").reset_index(drop=True)
X = train_df[FEATURE_COLUMNS]
y = train_df["y_target"].astype(int)

split = int(len(X) * 0.8)
Xtr, Xva = X.iloc[:split], X.iloc[split:]
ytr, yva = y.iloc[:split], y.iloc[split:]

def objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 250, 700),
        "depth": trial.suggest_int("depth", 4, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 2.0, 20.0),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 20, 200),
        "random_strength": trial.suggest_float("random_strength", 0.5, 3.0),
        "loss_function": "Logloss",
        "eval_metric": "PRAUC",
        "random_seed": CFG.RANDOM_STATE,
        "verbose": False,
    }
    tscv = TimeSeriesSplit(n_splits=3)
    scores = []
    for tr_idx, va_idx in tscv.split(Xtr):
        xxtr, xxva = Xtr.iloc[tr_idx], Xtr.iloc[va_idx]
        yytr, yyva = ytr.iloc[tr_idx], ytr.iloc[va_idx]
        if yytr.nunique() < 2 or yyva.nunique() < 2:
            continue
        n_pos = yytr.sum()
        n_neg = len(yytr) - n_pos
        params["scale_pos_weight"] = float(np.sqrt(n_neg / max(1, n_pos)))
        m = CatBoostClassifier(**params)
        m.fit(xxtr, yytr, cat_features=[c for c in cat_cols if c in xxtr.columns])
        p = m.predict_proba(xxva)[:, 1]
        scores.append(average_precision_score(yyva, p))
    return float(np.mean(scores)) if scores else 0.0

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=12, show_progress_bar=False)

best = study.best_params
n_pos = ytr.sum()
n_neg = len(ytr) - n_pos
best.update({
    "loss_function": "Logloss",
    "eval_metric": "PRAUC",
    "scale_pos_weight": float(np.sqrt(n_neg / max(1, n_pos))),
    "random_seed": CFG.RANDOM_STATE,
    "verbose": False,
})

model = CatBoostClassifier(**best)
model.fit(Xtr, ytr, cat_features=[c for c in cat_cols if c in Xtr.columns])
calibrator = CalibratedClassifierCV(model, method="sigmoid", cv="prefit")
calibrator.fit(Xva, yva)

p = calibrator.predict_proba(Xva)[:, 1]
pred = (p >= 0.5).astype(int)

print(roc_auc_score(yva, p), average_precision_score(yva, p), brier_score_loss(yva, p))
print(classification_report(yva, pred, zero_division=0))

Score Latest Snapshot + NBA

In [ ]:
latest_date = scoring_dates.max()
# Reuse earlier logic to build current snapshot, then engineer features similarly
# For brevity in notebook use:
score_df = train_df[train_df["scoring_date"] == latest_date].copy()

score_df["base_propensity"] = calibrator.predict_proba(score_df[FEATURE_COLUMNS])[:, 1]
ratio = score_df["latest_closing_stock"].fillna(0) / CFG.STOCK_THRESHOLD
score_df["adjusted_propensity"] = score_df["base_propensity"] * np.minimum(1.0, np.maximum(0.0, ratio.values)) * 100
score_df["base_propensity"] = score_df["base_propensity"] * 100

def pick_reason(row):
    if row.get("same_sku_conversion_rate", 0) >= 0.50:
        return "high historical same-SKU conversion"
    if row.get("historical_conversion_rate", 0) >= 0.40:
        return "strong overall historical conversion"
    if row.get("scheme_sensitivity_index", 0) >= 0.40:
        return "high scheme sensitivity"
    if row.get("latest_closing_stock", 0) >= CFG.STOCK_THRESHOLD:
        return "healthy local stock availability"
    return "balanced multi-factor signal"

score_df["top_reason"] = score_df.apply(pick_reason, axis=1)
score_df = score_df.sort_values("adjusted_propensity", ascending=False)
score_df[["account_name", "sku_code", "base_propensity", "adjusted_propensity", "top_reason"]].head(20)

In [ ]:
import gc
import math
import logging
import traceback
import warnings
from dataclasses import dataclass
from enum import Enum
from typing import Any, Dict, Iterable, List, Optional, Tuple

import boto3
import ijson
import numpy as np
import optuna
import pandas as pd
from textblob import TextBlob
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, classification_report, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.neighbors import BallTree, NearestNeighbors
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore", category=UserWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ============================================================
# LOGGING
# ============================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger("predictive_targeting_priority_upgrade")


# ============================================================
# CONFIG
# ============================================================

@dataclass
class Config:
    S3_BUCKET: str = "***"
    S3_PREFIX: str = "***"

    ACCOUNTS_PATH: str = ""
    CALLS_PATH: str = ""
    STOCK_PATH: str = ""

    RANDOM_STATE: int = 42

    STREAM_LIMIT: Optional[int] = None
    MAX_SCORING_WEEKS: int = 12
    MAX_TOP_SKUS_PER_POOL: int = 3
    USE_POOL_TOP_SKUS: bool = False
    RECENT_ACCOUNT_LOOKBACK_DAYS: int = 180
    LOOKBACK_DAYS: int = 180
    DISPENSER_TARGET_DAYS: int = 7
    SCORING_FREQ: str = "W-MON"
    FEATURE_CHUNK_SIZE: int = 5000

    GEO_RADIUS_KM: float = 10.0
    STOCK_THRESHOLD: float = 50.0

    COLD_MIN_VISITS: int = 3
    EPSILON_EXPLORE: float = 0.10

    VACCINES_BU: str = "VACCINES DOMESTIC"
    NUTRITION_BUS: Tuple[str, ...] = ("FAN", "LEAP_GT")
    VACCINES_ITEMS_KEY: str = "orderLineItems"
    NUTRITION_ITEMS_KEY: str = "nutritionOrderLineItems"

    POB_POSITIVE_STATUSES: Tuple[str, ...] = ("delivered", "partially fulfilled")
    POB_CANCELLED_STATUS: str = "cancelled"

    OUTPUT_FILENAME: str = "final_predictive_targeting_output_priority_upgrade.csv"
    EVAL_OUTPUT_FILENAME: str = "final_model_evaluation_priority_upgrade.csv"
    FEATURE_IMPORTANCE_FILENAME: str = "feature_importance_priority_upgrade.csv"
    CALIBRATION_FILENAME: str = "calibration_diagnostics_priority_upgrade.csv"

    CATEGORICAL_FEATURES: Tuple[str, ...] = (
        "account_type",
        "state",
        "region",
        "zone",
        "pool_name",
        "territory_name",
        "town_type",
        "specialty",
        "practice_type",
        "account_potential",
        "scheme_name_last_30d",
        "call_mode_last",
        "call_type_last",
        "time_of_day_last",
    )


CFG = Config(
    ACCOUNTS_PATH=f"***",
    CALLS_PATH=f"***",
    STOCK_PATH=f"***",
)


class MongoKeys(str, Enum):
    OID = "$oid"
    DATE = "$date"
    NUMBER_LONG = "$numberLong"


class AccountType(str, Enum):
    DOCTOR = "Doctor"
    RETAILER = "Retailer"
    INSTITUTION = "Institution"
    DISTRIBUTOR = "Distributor"
    UNKNOWN = "UNKNOWN"


# ============================================================
# UTILS
# ============================================================

class DataUtils:
    @staticmethod
    def extract_oid(field: Any) -> Optional[str]:
        if isinstance(field, dict) and MongoKeys.OID.value in field:
            return field[MongoKeys.OID.value]
        return None if field is None else str(field)

    @staticmethod
    def extract_date(field: Any) -> pd.Timestamp:
        if not field:
            return pd.NaT
        if isinstance(field, dict) and MongoKeys.DATE.value in field:
            val = field[MongoKeys.DATE.value]
            if isinstance(val, dict) and MongoKeys.NUMBER_LONG.value in val:
                return pd.to_datetime(int(val[MongoKeys.NUMBER_LONG.value]), unit="ms", utc=True, errors="coerce")
            return pd.to_datetime(val, utc=True, errors="coerce")
        return pd.to_datetime(field, utc=True, errors="coerce")

    @staticmethod
    def stream_json_file(filepath: str, limit: Optional[int] = None) -> Iterable[dict]:
        logger.info(f"Streaming file: {filepath}")
        if filepath.startswith("s3://"):
            bucket, key = filepath[5:].split("/", 1)
            s3 = boto3.client("s3")
            response = s3.get_object(Bucket=bucket, Key=key)
            stream = response["Body"]
        else:
            stream = open(filepath, "rb")
        try:
            for i, item in enumerate(ijson.items(stream, "item")):
                if limit is not None and i >= limit:
                    break
                yield item
        finally:
            stream.close()

    @staticmethod
    def summarize_df(df: pd.DataFrame, name: str):
        logger.info(f"{name}: shape={df.shape}")

    @staticmethod
    def reduce_memory(df: pd.DataFrame) -> pd.DataFrame:
        if df.empty:
            return df
        for col in df.columns:
            if pd.api.types.is_integer_dtype(df[col]):
                df[col] = pd.to_numeric(df[col], downcast="integer")
            elif pd.api.types.is_float_dtype(df[col]):
                df[col] = pd.to_numeric(df[col], downcast="float")
        return df

    @staticmethod
    def gc():
        gc.collect()

    @staticmethod
    def to_rad(lon: pd.Series, lat: pd.Series) -> np.ndarray:
        return np.column_stack([
            np.radians(lat.astype(float).values),
            np.radians(lon.astype(float).values)
        ])


# ============================================================
# CALIBRATION
# ============================================================

class ProbabilityCalibrator:
    def __init__(self):
        self.lr = None
        self.is_fitted = False

    def fit(self, raw_probs: np.ndarray, y_true: pd.Series):
        x = np.asarray(raw_probs).reshape(-1, 1)
        y = np.asarray(y_true).astype(int)

        if len(np.unique(y)) < 2:
            logger.warning("Calibration skipped: validation target has no variance.")
            self.is_fitted = False
            return self

        self.lr = LogisticRegression(random_state=CFG.RANDOM_STATE, max_iter=1000)
        self.lr.fit(x, y)
        self.is_fitted = True
        return self

    def predict(self, raw_probs: np.ndarray) -> np.ndarray:
        raw_probs = np.asarray(raw_probs).reshape(-1, 1)
        if self.is_fitted and self.lr is not None:
            return self.lr.predict_proba(raw_probs)[:, 1]
        return raw_probs.ravel()


# ============================================================
# EVALUATION / DIAGNOSTICS
# ============================================================

class ModelDiagnostics:
    @staticmethod
    def top_k_metrics(y_true: pd.Series, y_prob: np.ndarray, k_list: List[float] = [0.01, 0.05, 0.10, 0.20]) -> pd.DataFrame:
        df = pd.DataFrame({"y": y_true.values, "p": y_prob}).sort_values("p", ascending=False).reset_index(drop=True)
        baseline = df["y"].mean()
        rows = []

        for k in k_list:
            n = max(1, int(len(df) * k))
            sub = df.head(n)
            precision_k = sub["y"].mean() if len(sub) else 0.0
            recall_k = sub["y"].sum() / max(1, df["y"].sum())
            lift_k = precision_k / baseline if baseline > 0 else 0.0

            rows.append({
                "top_fraction": k,
                "top_n": n,
                "baseline_rate": baseline,
                "precision_at_k": precision_k,
                "recall_at_k": recall_k,
                "lift_at_k": lift_k,
            })

        return pd.DataFrame(rows)

    @staticmethod
    def decile_gains_table(y_true: pd.Series, y_prob: np.ndarray, n_bins: int = 10) -> pd.DataFrame:
        df = pd.DataFrame({"y": y_true.values, "p": y_prob}).sort_values("p", ascending=False).reset_index(drop=True)
        df["decile"] = pd.qcut(np.arange(len(df)), q=n_bins, labels=False, duplicates="drop") + 1

        baseline = df["y"].mean()
        rows = []
        cum_pos = 0
        total_pos = df["y"].sum()

        for d in sorted(df["decile"].unique()):
            sub = df[df["decile"] == d]
            pos = sub["y"].sum()
            rate = sub["y"].mean() if len(sub) else 0.0
            lift = rate / baseline if baseline > 0 else 0.0
            cum_pos += pos
            rows.append({
                "decile": int(d),
                "records": len(sub),
                "positives": int(pos),
                "conversion_rate": rate,
                "lift": lift,
                "cumulative_positives": int(cum_pos),
                "cumulative_recall": cum_pos / max(1, total_pos),
            })

        return pd.DataFrame(rows)

    @staticmethod
    def calibration_table(y_true: pd.Series, y_prob: np.ndarray, n_bins: int = 10) -> pd.DataFrame:
        df = pd.DataFrame({"y": y_true.values, "p": y_prob}).copy()
        df["bin"] = pd.qcut(df["p"].rank(method="first"), q=n_bins, duplicates="drop")
        out = (
            df.groupby("bin", observed=False)
            .agg(
                avg_predicted_prob=("p", "mean"),
                observed_rate=("y", "mean"),
                count=("y", "size"),
                positives=("y", "sum"),
            )
            .reset_index()
        )
        out["calibration_gap"] = out["avg_predicted_prob"] - out["observed_rate"]
        return out

    @staticmethod
    def score_distribution(y_prob: np.ndarray) -> pd.DataFrame:
        s = pd.Series(y_prob)
        return pd.DataFrame({
            "metric": ["min", "p01", "p05", "p10", "p25", "p50", "p75", "p90", "p95", "p99", "max", "mean"],
            "value": [
                s.min(),
                s.quantile(0.01),
                s.quantile(0.05),
                s.quantile(0.10),
                s.quantile(0.25),
                s.quantile(0.50),
                s.quantile(0.75),
                s.quantile(0.90),
                s.quantile(0.95),
                s.quantile(0.99),
                s.max(),
                s.mean(),
            ]
        })

    @staticmethod
    def mapping_quality(scored_df: pd.DataFrame) -> pd.DataFrame:
        total = len(scored_df)
        exact_mapping = scored_df["mapped_distributor_id"].notna().sum() if "mapped_distributor_id" in scored_df.columns else 0
        zero_stock = (scored_df["latest_closing_stock"].fillna(0) <= 0).sum() if "latest_closing_stock" in scored_df.columns else 0
        low_stock = (
            ((scored_df["latest_closing_stock"].fillna(0) > 0) &
             (scored_df["latest_closing_stock"].fillna(0) < CFG.STOCK_THRESHOLD)).sum()
            if "latest_closing_stock" in scored_df.columns else 0
        )

        rows = [
            {"metric": "total_scored_rows", "value": total},
            {"metric": "exact_distributor_mapping_count", "value": exact_mapping},
            {"metric": "exact_distributor_mapping_rate", "value": exact_mapping / max(1, total)},
            {"metric": "fallback_or_unmapped_count", "value": total - exact_mapping},
            {"metric": "fallback_or_unmapped_rate", "value": (total - exact_mapping) / max(1, total)},
            {"metric": "zero_stock_count", "value": zero_stock},
            {"metric": "zero_stock_rate", "value": zero_stock / max(1, total)},
            {"metric": "low_stock_count", "value": low_stock},
            {"metric": "low_stock_rate", "value": low_stock / max(1, total)},
        ]
        return pd.DataFrame(rows)


# ============================================================
# PARSERS
# ============================================================

class AccountsParser:
    @staticmethod
    def _std_type(raw: Optional[str]) -> str:
        if not raw:
            return AccountType.UNKNOWN.value
        raw = str(raw).strip()
        if raw in {"Hospital", "NursingHome", "MaternityHome", "Clinic", "Institution"}:
            return AccountType.INSTITUTION.value
        if raw in {"Distributor", "Stockist"}:
            return AccountType.DISTRIBUTOR.value
        if raw in {"Doctor", "Retailer"}:
            return raw
        return raw

    def parse(self, stream: Iterable[dict]) -> pd.DataFrame:
        rows = []
        for doc in stream:
            locations = doc.get("locations") or []
            effective_active = 0
            pool_name = "UNKNOWN_POOL"
            territory_name = "UNKNOWN_TERRITORY"
            region = "UNKNOWN_REGION"
            zone = "UNKNOWN_ZONE"
            town_type = "UNKNOWN_TOWN"
            business_units = []

            for loc in locations:
                if not isinstance(loc, dict):
                    continue
                if str(loc.get("status", "")).lower() == "active":
                    effective_active = 1
                    pool_name = loc.get("poolName", pool_name)
                    territory_name = loc.get("name", territory_name)
                    business_units.extend(loc.get("businessUnits", []) or [])
                    h = loc.get("locationHierarchy", {}) or {}
                    region = h.get("Region", region)
                    zone = h.get("Zone", zone)

            if territory_name == "UNKNOWN_TERRITORY":
                location_dict = doc.get("location") or {}
                if isinstance(location_dict, dict):
                    for _, loc_data in location_dict.items():
                        if isinstance(loc_data, dict):
                            territory_name = loc_data.get("name", territory_name)
                            pool_name = loc_data.get("poolName", pool_name)
                            town_type = loc_data.get("townType", town_type)
                            h = loc_data.get("locationHierarchy", {}) or {}
                            region = h.get("Region", region)
                            zone = h.get("Zone", zone)
                            break

            geo = doc.get("geoLocation") or {}
            coords = geo.get("coordinates", [None, None])
            lon = float(coords[0]) if len(coords) > 0 and coords[0] is not None else np.nan
            lat = float(coords[1]) if len(coords) > 1 and coords[1] is not None else np.nan

            info = doc.get("information") or {}
            qc = doc.get("qualityCheck") or {}
            addr = doc.get("address") or {}

            account_type = self._std_type(doc.get("accountType"))
            patient_load = 0.0
            account_potential = "UNKNOWN"
            specialty = "UNKNOWN"
            practice_type = "UNKNOWN"

            if account_type == AccountType.DOCTOR.value:
                patient_load = float((info.get("newBornBabiesPerMonth") or 0) + (info.get("infantsSeenPerDayAgeGroup6mTo2y") or 0))
                account_potential = info.get("potential", "UNKNOWN") or "UNKNOWN"
                specialty = info.get("specialty", "UNKNOWN") or "UNKNOWN"
                practice_type = info.get("practiceType", "UNKNOWN") or "UNKNOWN"
            elif account_type in {AccountType.RETAILER.value, AccountType.INSTITUTION.value}:
                patient_load = float(info.get("monthlyNutritionProductsSold") or 0)
                account_potential = info.get("nutritionPotentialClassification", "UNKNOWN") or "UNKNOWN"

            rows.append({
                "account_id": DataUtils.extract_oid(doc.get("_id")),
                "account_name": doc.get("name") if not isinstance(doc.get("name"), dict) else "Unknown",
                "account_type": account_type,
                "effective_active_flag": effective_active,
                "has_erp_code": int(bool(doc.get("erpCode"))),
                "state": addr.get("state") or "UNKNOWN",
                "city": addr.get("city") or "UNKNOWN",
                "region": region,
                "zone": zone,
                "pool_name": pool_name,
                "territory_name": territory_name,
                "town_type": town_type,
                "business_units": "|".join(sorted(set([str(x) for x in business_units]))),
                "longitude": lon,
                "latitude": lat,
                "profile_quality_score": float(qc.get("Score") or 0.0),
                "specialty": specialty,
                "practice_type": practice_type,
                "patient_load": patient_load,
                "account_potential": account_potential,
                "account_added_on": DataUtils.extract_date(doc.get("addedOn")),
            })

        df = pd.DataFrame(rows)
        if not df.empty:
            df["account_added_on"] = pd.to_datetime(df["account_added_on"], utc=True, errors="coerce")
        df = DataUtils.reduce_memory(df)
        DataUtils.summarize_df(df, "accounts_df")
        return df


class CallsParser:
    @staticmethod
    def _resolve_line_items(doc: dict) -> Tuple[List[dict], str]:
        account = doc.get("account") or {}
        location = account.get("location") or {}
        bu_list = location.get("businessUnits", []) if isinstance(location, dict) else []
        bu_set = {str(x).upper() for x in bu_list}

        if CFG.VACCINES_BU in bu_set:
            key = CFG.VACCINES_ITEMS_KEY
        elif any(x in bu_set for x in CFG.NUTRITION_BUS):
            key = CFG.NUTRITION_ITEMS_KEY
        else:
            key = CFG.NUTRITION_ITEMS_KEY if doc.get(CFG.NUTRITION_ITEMS_KEY) else CFG.VACCINES_ITEMS_KEY

        return doc.get(key) or [], key

    def parse(self, stream: Iterable[dict]) -> Tuple[pd.DataFrame, pd.DataFrame]:
        headers, lines = [], []

        for doc in stream:
            call_id = DataUtils.extract_oid(doc.get("_id"))
            call_date = DataUtils.extract_date(doc.get("date"))
            account = doc.get("account") or {}
            account_id = DataUtils.extract_oid(account.get("accountId"))
            account_type = account.get("accountType") or "UNKNOWN"

            campaigns = doc.get("campaigns") or []
            promo_inputs = doc.get("marketingInputsUtilized") or []
            remarks = doc.get("conversationRemarks") or ""
            sentiment = TextBlob(remarks).sentiment.polarity if remarks else 0.0
            team = doc.get("accompanyingTeamMembers") or []
            pob_status = str(doc.get("pobStatus") or "").strip().lower()

            geo = doc.get("geoLocation") or {}
            coords = geo.get("coordinates", [None, None])

            headers.append({
                "call_id": call_id,
                "account_id": account_id,
                "account_type_in_call": account_type,
                "call_date": call_date,
                "call_mode": doc.get("callMode") or "UNKNOWN",
                "call_type": doc.get("callType") or "UNKNOWN",
                "call_time_of_day": doc.get("time") or "UNKNOWN",
                "scheme_name": (doc.get("scheme") or "").strip() or "NO_SCHEME",
                "campaign_count": len(campaigns),
                "promo_input_count": len(promo_inputs),
                "remarks_sentiment": float(sentiment),
                "manager_present_flag": int(len(team) > 0),
                "distance_from_account": float(doc.get("distanceFromAccountLocation") or 0.0),
                "pob_status": pob_status,
                "call_longitude": float(coords[0]) if len(coords) > 0 and coords[0] is not None else np.nan,
                "call_latitude": float(coords[1]) if len(coords) > 1 and coords[1] is not None else np.nan,
            })

            line_items, items_key = self._resolve_line_items(doc)
            for item in line_items:
                fulfillment = item.get("fulfillmentAccount") or {}
                lines.append({
                    "call_id": call_id,
                    "account_id": account_id,
                    "call_date": call_date,
                    "account_type_in_call": account_type,
                    "sku_code": item.get("skuCode"),
                    "pob_quantity": float(item.get("quantity") or 0.0),
                    "quantity_cancelled": float(item.get("quantityCancelled") or 0.0),
                    "fulfillment_account_id": DataUtils.extract_oid(fulfillment.get("accountId")),
                    "items_key_used": items_key,
                })

        h = pd.DataFrame(headers)
        l = pd.DataFrame(lines)

        for df in (h, l):
            if not df.empty and "call_date" in df.columns:
                df["call_date"] = pd.to_datetime(df["call_date"], utc=True, errors="coerce")

        h = DataUtils.reduce_memory(h)
        l = DataUtils.reduce_memory(l)

        DataUtils.summarize_df(h, "calls_header_df")
        DataUtils.summarize_df(l, "calls_line_df")
        return h, l


class StockParser:
    def parse(self, stream: Iterable[dict]) -> pd.DataFrame:
        rows = []
        for doc in stream:
            account = doc.get("account") or {}
            location = account.get("location") or {}
            pool_name = location.get("poolName", "UNKNOWN_POOL") if isinstance(location, dict) else "UNKNOWN_POOL"

            for item in doc.get("inventoryBook") or []:
                rows.append({
                    "distributor_id": DataUtils.extract_oid(account.get("accountId")),
                    "pool_name": pool_name,
                    "statement_date": DataUtils.extract_date(doc.get("statementDate")),
                    "uploaded_on": DataUtils.extract_date(doc.get("uploadedOn")),
                    "sku_code": item.get("skuCode"),
                    "opening_stock": float(item.get("opening") or 0.0),
                    "purchase_qty": float(item.get("purchase") or 0.0),
                    "secondary_sales_qty": float(item.get("sales") or 0.0),
                    "in_transit": float(item.get("inTransit") or 0.0),
                    "closing_stock": float(item.get("closing") or 0.0),
                })

        df = pd.DataFrame(rows)
        if not df.empty:
            df["statement_date"] = pd.to_datetime(df["statement_date"], utc=True, errors="coerce")
            df["uploaded_on"] = pd.to_datetime(df["uploaded_on"], utc=True, errors="coerce")
        df = DataUtils.reduce_memory(df)
        DataUtils.summarize_df(df, "stock_df")
        return df


# ============================================================
# NORMALIZATION
# ============================================================

class EntityResolver:
    @staticmethod
    def build_calls_table(headers: pd.DataFrame, lines: pd.DataFrame, accounts_df: pd.DataFrame) -> pd.DataFrame:
        if lines.empty:
            return pd.DataFrame()

        calls = lines.merge(
            headers,
            on=["call_id", "account_id", "call_date", "account_type_in_call"],
            how="left"
        ).merge(
            accounts_df,
            on="account_id",
            how="left",
            suffixes=("", "_acct")
        )

        calls["account_type"] = calls["account_type"].fillna(calls["account_type_in_call"]).fillna("UNKNOWN")
        calls = DataUtils.reduce_memory(calls)
        DataUtils.summarize_df(calls, "calls_df")
        return calls

    @staticmethod
    def build_fulfillment_mapping(calls_df: pd.DataFrame) -> pd.DataFrame:
        if calls_df.empty:
            return pd.DataFrame(columns=["account_id", "sku_code", "mapped_distributor_id"])

        valid = calls_df[
            calls_df["fulfillment_account_id"].notna() &
            calls_df["sku_code"].notna() &
            calls_df["pob_status"].isin(CFG.POB_POSITIVE_STATUSES)
        ]

        if valid.empty:
            return pd.DataFrame(columns=["account_id", "sku_code", "mapped_distributor_id"])

        mapping = (
            valid.groupby(["account_id", "sku_code", "fulfillment_account_id"])
            .size()
            .reset_index(name="cnt")
            .sort_values(["account_id", "sku_code", "cnt"], ascending=[True, True, False])
            .groupby(["account_id", "sku_code"])
            .head(1)
            .rename(columns={"fulfillment_account_id": "mapped_distributor_id"})
        )[["account_id", "sku_code", "mapped_distributor_id"]]

        mapping = DataUtils.reduce_memory(mapping)
        DataUtils.summarize_df(mapping, "fulfillment_map")
        return mapping


# ============================================================
# SNAPSHOTS
# ============================================================

class CandidateGenerator:
    @staticmethod
    def build_training_ranges(calls_df: pd.DataFrame, stock_df: pd.DataFrame) -> Tuple[pd.Timestamp, pd.Timestamp]:
        min_call = calls_df["call_date"].min()
        max_call = calls_df["call_date"].max()
        min_stock = stock_df["statement_date"].min() if not stock_df.empty else min_call
        start = max(min_call, min_stock) + pd.Timedelta(days=CFG.LOOKBACK_DAYS)
        end = max_call - pd.Timedelta(days=CFG.DISPENSER_TARGET_DAYS)
        return start, end

    @staticmethod
    def generate_scoring_dates(start_date: pd.Timestamp, end_date: pd.Timestamp) -> pd.DatetimeIndex:
        dates = pd.date_range(
            pd.to_datetime(start_date, utc=True).normalize(),
            pd.to_datetime(end_date, utc=True).normalize(),
            freq=CFG.SCORING_FREQ,
            tz="UTC"
        )
        if len(dates) > CFG.MAX_SCORING_WEEKS:
            dates = dates[-CFG.MAX_SCORING_WEEKS:]
        return dates

    @staticmethod
    def top_skus_by_pool(calls_df: pd.DataFrame, top_n: int) -> pd.DataFrame:
        if calls_df.empty:
            return pd.DataFrame(columns=["pool_name", "sku_code"])
        x = (
            calls_df[calls_df["sku_code"].notna()]
            .groupby(["pool_name", "sku_code"])
            .size()
            .reset_index(name="cnt")
            .sort_values(["pool_name", "cnt"], ascending=[True, False])
        )
        x["rk"] = x.groupby("pool_name").cumcount() + 1
        return x[x["rk"] <= top_n][["pool_name", "sku_code"]]

    @staticmethod
    def recent_account_skus(calls_df: pd.DataFrame, scoring_date: pd.Timestamp) -> pd.DataFrame:
        lo = scoring_date - pd.Timedelta(days=CFG.LOOKBACK_DAYS)
        hist = calls_df[
            (calls_df["call_date"] < scoring_date) &
            (calls_df["call_date"] >= lo) &
            calls_df["sku_code"].notna()
        ]
        return hist[["account_id", "sku_code"]].drop_duplicates()

    @staticmethod
    def build_dispenser_snapshots(
        accounts_df: pd.DataFrame,
        calls_df: pd.DataFrame,
        stock_df: pd.DataFrame,
        scoring_dates: pd.DatetimeIndex
    ) -> pd.DataFrame:
        recent_cutoff = calls_df["call_date"].max() - pd.Timedelta(days=CFG.RECENT_ACCOUNT_LOOKBACK_DAYS)
        recent_accounts = calls_df.loc[calls_df["call_date"] >= recent_cutoff, "account_id"].dropna().unique()

        active = accounts_df[
            (accounts_df["effective_active_flag"] == 1) &
            (accounts_df["account_type"].isin([AccountType.RETAILER.value, AccountType.INSTITUTION.value])) &
            (accounts_df["account_id"].isin(recent_accounts))
        ].copy()

        if active.empty:
            return pd.DataFrame()

        base_cols = [
            "account_id", "account_name", "account_type", "pool_name", "territory_name", "region", "zone",
            "state", "city", "town_type", "specialty", "practice_type", "patient_load",
            "account_potential", "profile_quality_score", "longitude", "latitude",
            "account_added_on", "has_erp_code", "effective_active_flag"
        ]

        pool_top = CandidateGenerator.top_skus_by_pool(calls_df, top_n=CFG.MAX_TOP_SKUS_PER_POOL) if CFG.USE_POOL_TOP_SKUS else None
        results = []

        for t in scoring_dates:
            base = active[base_cols].copy()
            base["scoring_date"] = t

            tmp = base.merge(
                CandidateGenerator.recent_account_skus(calls_df, t),
                on="account_id",
                how="inner"
            )

            if CFG.USE_POOL_TOP_SKUS and pool_top is not None and not pool_top.empty:
                tmp2 = base.merge(pool_top, on="pool_name", how="inner")
                tmp = pd.concat([tmp, tmp2], ignore_index=True)
                del tmp2

            if not tmp.empty:
                tmp = tmp.drop_duplicates(["account_id", "sku_code", "scoring_date"])
                results.append(tmp)

            del base, tmp
            DataUtils.gc()

        snapshots = pd.concat(results, ignore_index=True) if results else pd.DataFrame()
        snapshots = DataUtils.reduce_memory(snapshots)
        DataUtils.summarize_df(snapshots, "snapshots")
        return snapshots


# ============================================================
# TARGETS
# ============================================================

class TargetBuilderDispenser:
    @staticmethod
    def build(snapshots: pd.DataFrame, calls_df: pd.DataFrame) -> pd.DataFrame:
        if snapshots.empty:
            return snapshots.assign(y_target=0)

        pos = calls_df[
            calls_df["pob_status"].isin(CFG.POB_POSITIVE_STATUSES) &
            calls_df["sku_code"].notna()
        ][["account_id", "sku_code", "call_date"]].drop_duplicates().copy()

        pos["call_date"] = pd.to_datetime(pos["call_date"], utc=True, errors="coerce")
        pos = pos.dropna(subset=["call_date"])

        pos_grouped = {
            key: grp["call_date"].dt.tz_convert("UTC").dt.tz_localize(None).values
            for key, grp in pos.groupby(["account_id", "sku_code"])
        }

        scoring_dates_np = (
            pd.to_datetime(snapshots["scoring_date"], utc=True, errors="coerce")
            .dt.tz_convert("UTC")
            .dt.tz_localize(None)
            .values
        )

        def label_row(acc, sku, lo):
            arr = pos_grouped.get((acc, sku))
            if arr is None or len(arr) == 0:
                return 0
            hi = lo + np.timedelta64(CFG.DISPENSER_TARGET_DAYS, "D")
            return int(((arr > lo) & (arr <= hi)).any())

        y = np.fromiter(
            (
                label_row(a, s, lo)
                for a, s, lo in zip(
                    snapshots["account_id"].values,
                    snapshots["sku_code"].values,
                    scoring_dates_np
                )
            ),
            dtype=np.int8,
            count=len(snapshots)
        )

        out = snapshots.copy()
        out["y_target"] = y
        logger.info(f"Target positive rate: {out['y_target'].mean():.4f}")
        return out


# ============================================================
# FEATURES
# ============================================================

class SupplyAsOfJoiner:
    @staticmethod
    def add(snapshot_df: pd.DataFrame, stock_df: pd.DataFrame, fulfill_map: pd.DataFrame) -> pd.DataFrame:
        out = snapshot_df.copy()
        out["snapshot_row"] = np.arange(len(out), dtype=np.int32)
        out = out.merge(fulfill_map, on=["account_id", "sku_code"], how="left")

        if stock_df.empty:
            for c in [
                "latest_opening_stock", "latest_purchase_qty", "latest_secondary_sales_qty",
                "latest_in_transit", "latest_closing_stock"
            ]:
                out[c] = 0.0
            out["sell_through_rate"] = 0.0
            out["stock_cover_days"] = 999.0
            out["zero_stock_flag"] = 1
            out["low_stock_flag"] = 0
            return out

        exact = out[out["mapped_distributor_id"].notna()][["snapshot_row", "mapped_distributor_id", "sku_code", "scoring_date"]]
        if not exact.empty:
            tmp = exact.merge(
                stock_df.rename(columns={"distributor_id": "mapped_distributor_id"}),
                on=["mapped_distributor_id", "sku_code"],
                how="left"
            )
            tmp = tmp[tmp["statement_date"] <= tmp["scoring_date"]]
            if not tmp.empty:
                tmp = tmp.sort_values(["snapshot_row", "statement_date", "uploaded_on"]).groupby("snapshot_row").tail(1)
                tmp = tmp[["snapshot_row", "opening_stock", "purchase_qty", "secondary_sales_qty", "in_transit", "closing_stock"]]
                tmp = tmp.rename(columns={
                    "opening_stock": "distributor_opening_stock",
                    "purchase_qty": "distributor_purchase_qty",
                    "secondary_sales_qty": "distributor_secondary_sales_qty",
                    "in_transit": "distributor_in_transit",
                    "closing_stock": "distributor_closing_stock",
                })
                out = out.merge(tmp, on="snapshot_row", how="left")
            del tmp
            DataUtils.gc()

        tmp2 = out[["snapshot_row", "pool_name", "sku_code", "scoring_date"]].merge(
            stock_df, on=["pool_name", "sku_code"], how="left"
        )
        tmp2 = tmp2[tmp2["statement_date"] <= tmp2["scoring_date"]]
        if not tmp2.empty:
            tmp2 = tmp2.sort_values(["snapshot_row", "statement_date", "uploaded_on"]).groupby("snapshot_row").tail(1)
            tmp2 = tmp2[["snapshot_row", "opening_stock", "purchase_qty", "secondary_sales_qty", "in_transit", "closing_stock"]]
            tmp2 = tmp2.rename(columns={
                "opening_stock": "pool_opening_stock",
                "purchase_qty": "pool_purchase_qty",
                "secondary_sales_qty": "pool_secondary_sales_qty",
                "in_transit": "pool_in_transit",
                "closing_stock": "pool_closing_stock",
            })
            out = out.merge(tmp2, on="snapshot_row", how="left")

        for a, b, c in [
            ("latest_opening_stock", "distributor_opening_stock", "pool_opening_stock"),
            ("latest_purchase_qty", "distributor_purchase_qty", "pool_purchase_qty"),
            ("latest_secondary_sales_qty", "distributor_secondary_sales_qty", "pool_secondary_sales_qty"),
            ("latest_in_transit", "distributor_in_transit", "pool_in_transit"),
            ("latest_closing_stock", "distributor_closing_stock", "pool_closing_stock"),
        ]:
            if b not in out.columns:
                out[b] = np.nan
            if c not in out.columns:
                out[c] = np.nan
            out[a] = out[b].fillna(out[c])

        total_avail = (
            out["latest_opening_stock"].fillna(0) +
            out["latest_purchase_qty"].fillna(0) +
            out["latest_in_transit"].fillna(0)
        )
        out["sell_through_rate"] = np.where(
            total_avail > 0,
            out["latest_secondary_sales_qty"].fillna(0) / total_avail,
            0.0
        )

        avg_daily = out["latest_secondary_sales_qty"].fillna(0) / 30.0
        out["stock_cover_days"] = np.where(avg_daily > 0, out["latest_closing_stock"].fillna(0) / avg_daily, 999.0)
        out["zero_stock_flag"] = (out["latest_closing_stock"].fillna(0) <= 0).astype(np.int8)
        out["low_stock_flag"] = (
            (out["latest_closing_stock"].fillna(0) > 0) &
            (out["latest_closing_stock"].fillna(0) < CFG.STOCK_THRESHOLD)
        ).astype(np.int8)

        out = out.drop(columns=["snapshot_row"], errors="ignore")
        return DataUtils.reduce_memory(out)


class HistoricalFeatureBuilder:
    def __init__(self, calls_df: pd.DataFrame, accounts_df: pd.DataFrame, stock_df: pd.DataFrame):
        self.calls_df = calls_df.sort_values("call_date").copy()
        self.accounts_df = accounts_df.copy()
        self.stock_df = stock_df.copy()

        self.by_account = {k: g for k, g in self.calls_df.groupby("account_id")}
        self.by_account_sku = {k: g for k, g in self.calls_df.groupby(["account_id", "sku_code"])}

    def _process_chunk(self, chunk_df: pd.DataFrame) -> pd.DataFrame:
        feats = []

        for idx, row in chunk_df.iterrows():
            acc = row["account_id"]
            sku = row["sku_code"]
            t = row["scoring_date"]

            hist = self.by_account.get(acc)
            if hist is not None:
                hist = hist[hist["call_date"] < t]
            else:
                hist = pd.DataFrame()

            hist_sku = self.by_account_sku.get((acc, sku))
            if hist_sku is not None:
                hist_sku = hist_sku[hist_sku["call_date"] < t]
            else:
                hist_sku = pd.DataFrame()

            if hist.empty:
                feats.append({
                    "row_id": idx,
                    "historical_visits": 0.0,
                    "visits_last_7d": 0.0,
                    "visits_last_30d": 0.0,
                    "visits_last_90d": 0.0,
                    "days_since_last_visit": 999.0,
                    "historical_positive_pobs": 0.0,
                    "historical_conversion_rate": 0.0,
                    "days_since_last_positive_pob": 999.0,
                    "avg_prior_pob_quantity": 0.0,
                    "same_sku_pitches_last_30d": 0.0,
                    "same_sku_positive_pobs": 0.0,
                    "same_sku_conversion_rate": 0.0,
                    "unique_skus_pitched_last_90d": 0.0,
                    "cancellation_count": 0.0,
                    "cancellation_ratio": 0.0,
                    "total_cancelled_units": 0.0,
                    "scheme_calls_last_30d": 0.0,
                    "scheme_positive_last_30d": 0.0,
                    "scheme_sensitivity_index": 0.0,
                    "campaign_count_last_30d": 0.0,
                    "promo_input_count_last_30d": 0.0,
                    "remarks_sentiment_avg_last_30d": 0.0,
                    "scheme_name_last_30d": "NO_SCHEME",
                    "call_mode_last": "UNKNOWN",
                    "call_type_last": "UNKNOWN",
                    "time_of_day_last": "UNKNOWN",
                })
                continue

            lo7 = t - pd.Timedelta(days=7)
            lo30 = t - pd.Timedelta(days=30)
            lo90 = t - pd.Timedelta(days=90)

            h7 = hist[hist["call_date"] >= lo7]
            h30 = hist[hist["call_date"] >= lo30]
            h90 = hist[hist["call_date"] >= lo90]
            hsku30 = hist_sku[hist_sku["call_date"] >= lo30] if not hist_sku.empty else pd.DataFrame()

            pos = hist[hist["pob_status"].isin(CFG.POB_POSITIVE_STATUSES)]
            pos_sku = hist_sku[hist_sku["pob_status"].isin(CFG.POB_POSITIVE_STATUSES)] if not hist_sku.empty else pd.DataFrame()
            canc = hist[hist["pob_status"] == CFG.POB_CANCELLED_STATUS]

            last_visit = hist["call_date"].max()
            last_pos = pos["call_date"].max() if not pos.empty else pd.NaT
            last30 = h30.sort_values("call_date")

            last_scheme = last30["scheme_name"].dropna().iloc[-1] if not last30.empty else "NO_SCHEME"
            last_mode = last30["call_mode"].dropna().iloc[-1] if not last30.empty else "UNKNOWN"
            last_type = last30["call_type"].dropna().iloc[-1] if not last30.empty else "UNKNOWN"
            last_tod = last30["call_time_of_day"].dropna().iloc[-1] if not last30.empty else "UNKNOWN"

            scheme_hist = hist[hist["scheme_name"] == last_scheme] if not hist.empty else hist
            scheme_pos = scheme_hist[scheme_hist["pob_status"].isin(CFG.POB_POSITIVE_STATUSES)] if not scheme_hist.empty else scheme_hist

            feats.append({
                "row_id": idx,
                "historical_visits": float(len(hist)),
                "visits_last_7d": float(len(h7)),
                "visits_last_30d": float(len(h30)),
                "visits_last_90d": float(len(h90)),
                "days_since_last_visit": float((t - last_visit).days) if pd.notna(last_visit) else 999.0,
                "historical_positive_pobs": float(len(pos)),
                "historical_conversion_rate": float(len(pos) / len(hist)) if len(hist) else 0.0,
                "days_since_last_positive_pob": float((t - last_pos).days) if pd.notna(last_pos) else 999.0,
                "avg_prior_pob_quantity": float(pos["pob_quantity"].mean()) if not pos.empty else 0.0,
                "same_sku_pitches_last_30d": float(len(hsku30)),
                "same_sku_positive_pobs": float(len(pos_sku)),
                "same_sku_conversion_rate": float(len(pos_sku) / len(hist_sku)) if len(hist_sku) else 0.0,
                "unique_skus_pitched_last_90d": float(h90["sku_code"].nunique()),
                "cancellation_count": float(len(canc)),
                "cancellation_ratio": float(len(canc) / len(hist)) if len(hist) else 0.0,
                "total_cancelled_units": float(hist["quantity_cancelled"].sum()) if not hist.empty else 0.0,
                "scheme_calls_last_30d": float(len(last30[last30["scheme_name"] != "NO_SCHEME"])),
                "scheme_positive_last_30d": float(len(last30[
                    (last30["scheme_name"] != "NO_SCHEME") &
                    (last30["pob_status"].isin(CFG.POB_POSITIVE_STATUSES))
                ])),
                "scheme_sensitivity_index": float(len(scheme_pos) / len(scheme_hist)) if len(scheme_hist) else 0.0,
                "campaign_count_last_30d": float(last30["campaign_count"].sum()) if not last30.empty else 0.0,
                "promo_input_count_last_30d": float(last30["promo_input_count"].sum()) if not last30.empty else 0.0,
                "remarks_sentiment_avg_last_30d": float(last30["remarks_sentiment"].mean()) if not last30.empty else 0.0,
                "scheme_name_last_30d": str(last_scheme),
                "call_mode_last": str(last_mode),
                "call_type_last": str(last_type),
                "time_of_day_last": str(last_tod),
            })

        return pd.DataFrame(feats)

    def add_history_chunked(self, snapshot_df: pd.DataFrame) -> pd.DataFrame:
        out = snapshot_df.copy().reset_index(drop=True)
        out["row_id"] = np.arange(len(out), dtype=np.int32)

        chunks = []
        for start in range(0, len(out), CFG.FEATURE_CHUNK_SIZE):
            end = min(start + CFG.FEATURE_CHUNK_SIZE, len(out))
            logger.info(f"Feature chunk {start}:{end}")
            chunk = out.iloc[start:end].copy()
            feat_df = self._process_chunk(chunk)
            chunks.append(feat_df)
            del chunk, feat_df
            DataUtils.gc()

        hist_feat = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()
        out = out.merge(hist_feat, on="row_id", how="left").drop(columns=["row_id"])
        return out

    def add_geo(self, snapshot_df: pd.DataFrame) -> pd.DataFrame:
        out = snapshot_df.copy()
        out["active_pharmacy_density"] = 0.0

        disp = self.accounts_df[
            (self.accounts_df["effective_active_flag"] == 1) &
            (self.accounts_df["account_type"].isin([AccountType.RETAILER.value, AccountType.INSTITUTION.value])) &
            self.accounts_df["longitude"].notna() &
            self.accounts_df["latitude"].notna()
        ]
        tgt = out[out["longitude"].notna() & out["latitude"].notna()]

        if not disp.empty and not tgt.empty:
            tree = BallTree(DataUtils.to_rad(disp["longitude"], disp["latitude"]), metric="haversine")
            counts = tree.query_radius(
                DataUtils.to_rad(tgt["longitude"], tgt["latitude"]),
                r=CFG.GEO_RADIUS_KM / 6371.0,
                count_only=True
            )
            out.loc[tgt.index, "active_pharmacy_density"] = counts

        return out

    def add_temporal_and_composite(self, snapshot_df: pd.DataFrame) -> pd.DataFrame:
        out = snapshot_df.copy()
        out["account_age_days"] = (out["scoring_date"] - out["account_added_on"]).dt.days.fillna(999).astype(float)
        out["is_month_end"] = (out["scoring_date"].dt.day >= 25).astype(np.int8)
        out["month"] = out["scoring_date"].dt.month.astype(np.int8)
        out["quarter"] = out["scoring_date"].dt.quarter.astype(np.int8)
        out["weekofyear"] = out["scoring_date"].dt.isocalendar().week.astype("int16")

        out["patient_load_to_pob_ratio"] = out["patient_load"].fillna(0) / (out["historical_positive_pobs"].fillna(0) + 1.0)
        out["effort_to_conversion_ratio"] = out["historical_visits"].fillna(0) / (out["historical_positive_pobs"].fillna(0) + 1.0)
        out["stock_x_same_sku_conv"] = out["latest_closing_stock"].fillna(0) * out["same_sku_conversion_rate"].fillna(0)
        out["stock_x_hist_conv"] = out["latest_closing_stock"].fillna(0) * out["historical_conversion_rate"].fillna(0)
        return out

    def run(self, snapshots: pd.DataFrame, fulfill_map: pd.DataFrame) -> pd.DataFrame:
        out = self.add_history_chunked(snapshots)
        out = SupplyAsOfJoiner.add(out, self.stock_df, fulfill_map)
        out = self.add_geo(out)
        out = self.add_temporal_and_composite(out)

        for c in out.select_dtypes(include=[np.number]).columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)
        for c in [x for x in CFG.CATEGORICAL_FEATURES if x in out.columns]:
            out[c] = out[c].fillna("UNKNOWN").astype(str)

        out = DataUtils.reduce_memory(out)
        return out


# ============================================================
# COLD START / EXPLORATION
# ============================================================

class ColdStartFeatureBuilder:
    STATIC_FEATURES = [
        "account_type", "state", "region", "zone", "pool_name", "territory_name",
        "town_type", "specialty", "practice_type", "account_potential",
        "patient_load", "profile_quality_score", "longitude", "latitude"
    ]

    @staticmethod
    def add(feature_df: pd.DataFrame) -> pd.DataFrame:
        out = feature_df.copy()
        out["cold_start_flag"] = (out["historical_visits"].fillna(0) < CFG.COLD_MIN_VISITS).astype(np.int8)
        out["cold_start_neighbor_conv_rate"] = 0.0
        out["cold_start_similarity_score"] = 0.0

        warm = out[out["cold_start_flag"] == 0].copy()
        cold = out[out["cold_start_flag"] == 1].copy()

        if warm.empty or cold.empty:
            return out

        xw = warm[ColdStartFeatureBuilder.STATIC_FEATURES].copy()
        xc = cold[ColdStartFeatureBuilder.STATIC_FEATURES].copy()

        comb = pd.concat([xw, xc], axis=0)
        for c in comb.columns:
            if comb[c].dtype == object:
                comb[c] = comb[c].fillna("UNKNOWN").astype("category").cat.codes
            else:
                comb[c] = pd.to_numeric(comb[c], errors="coerce").fillna(0.0)

        xw_arr = comb.iloc[:len(xw)].values
        xc_arr = comb.iloc[len(xw):].values

        nn = NearestNeighbors(n_neighbors=min(5, len(warm)), metric="euclidean")
        nn.fit(xw_arr)
        dists, idxs = nn.kneighbors(xc_arr)

        warm_rates = warm["historical_conversion_rate"].values
        for i, idx in enumerate(cold.index):
            out.at[idx, "cold_start_neighbor_conv_rate"] = float(np.mean(warm_rates[idxs[i]]))
            out.at[idx, "cold_start_similarity_score"] = float(1.0 / (1.0 + np.mean(dists[i])))

        return out


class ExplorationPolicy:
    @staticmethod
    def apply(df: pd.DataFrame, capacity: Optional[int] = None) -> pd.DataFrame:
        out = df.copy()
        out["targeting_mode"] = "Exploitation"

        if out.empty:
            return out

        n = len(out) if capacity is None else min(capacity, len(out))
        n_explore = max(1, int(math.ceil(n * CFG.EPSILON_EXPLORE)))

        cold = out[(out["cold_start_flag"] == 1) & (out["adjusted_propensity"] > 0)].copy()
        if cold.empty:
            return out

        chosen = cold.sort_values(
            ["cold_start_neighbor_conv_rate", "cold_start_similarity_score", "adjusted_propensity"],
            ascending=False
        ).head(n_explore)

        out.loc[chosen.index, "targeting_mode"] = "Exploration"
        return out


# ============================================================
# MODEL
# ============================================================

class DispenserModel:
    FEATURE_COLUMNS = [
        "account_type", "state", "region", "zone", "pool_name", "territory_name", "town_type",
        "specialty", "practice_type", "account_potential",
        "patient_load", "profile_quality_score", "has_erp_code", "effective_active_flag",
        "historical_visits", "visits_last_7d", "visits_last_30d", "visits_last_90d",
        "days_since_last_visit", "historical_positive_pobs", "historical_conversion_rate",
        "days_since_last_positive_pob", "avg_prior_pob_quantity",
        "same_sku_pitches_last_30d", "same_sku_positive_pobs", "same_sku_conversion_rate",
        "unique_skus_pitched_last_90d", "cancellation_count", "cancellation_ratio", "total_cancelled_units",
        "scheme_calls_last_30d", "scheme_positive_last_30d", "scheme_sensitivity_index",
        "campaign_count_last_30d", "promo_input_count_last_30d", "remarks_sentiment_avg_last_30d",
        "scheme_name_last_30d", "call_mode_last", "call_type_last", "time_of_day_last",
        "latest_opening_stock", "latest_purchase_qty", "latest_secondary_sales_qty", "latest_in_transit",
        "latest_closing_stock", "sell_through_rate", "stock_cover_days", "zero_stock_flag", "low_stock_flag",
        "active_pharmacy_density", "account_age_days", "is_month_end", "month", "quarter", "weekofyear",
        "patient_load_to_pob_ratio", "effort_to_conversion_ratio", "stock_x_same_sku_conv", "stock_x_hist_conv",
        "cold_start_flag", "cold_start_neighbor_conv_rate", "cold_start_similarity_score",
    ]

    def __init__(self):
        self.model = None
        self.calibrator = ProbabilityCalibrator()
        self.cat_cols = [c for c in CFG.CATEGORICAL_FEATURES if c in self.FEATURE_COLUMNS]
        self.validation_predictions_ = None
        self.validation_truth_ = None
        self.feature_importance_ = None

    def preprocess(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series]:
        X = df.copy()

        for c in self.FEATURE_COLUMNS:
            if c not in X.columns:
                X[c] = "UNKNOWN" if c in self.cat_cols else 0.0

        for c in self.cat_cols:
            X[c] = X[c].fillna("UNKNOWN").astype(str)
        for c in [x for x in self.FEATURE_COLUMNS if x not in self.cat_cols]:
            X[c] = pd.to_numeric(X[c], errors="coerce").fillna(0.0).astype(float)

        y = df["y_target"].astype(int) if "y_target" in df.columns else pd.Series(0, index=df.index)
        return X[self.FEATURE_COLUMNS], y

    def _objective(self, X_train: pd.DataFrame, y_train: pd.Series):
        def fn(trial):
            params = {
                "iterations": trial.suggest_int("iterations", 200, 500),
                "depth": trial.suggest_int("depth", 4, 7),
                "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.12, log=True),
                "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 2.0, 15.0),
                "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 20, 120),
                "random_strength": trial.suggest_float("random_strength", 0.5, 2.5),
                "loss_function": "Logloss",
                "eval_metric": "PRAUC",
                "random_seed": CFG.RANDOM_STATE,
                "verbose": False,
            }

            tscv = TimeSeriesSplit(n_splits=3)
            scores = []

            for tr_idx, va_idx in tscv.split(X_train):
                Xtr, Xva = X_train.iloc[tr_idx], X_train.iloc[va_idx]
                ytr, yva = y_train.iloc[tr_idx], y_train.iloc[va_idx]

                if ytr.nunique() < 2 or yva.nunique() < 2:
                    continue

                n_pos = ytr.sum()
                n_neg = len(ytr) - n_pos
                params["scale_pos_weight"] = float(np.sqrt(n_neg / max(1, n_pos)))

                m = CatBoostClassifier(**params)
                m.fit(Xtr, ytr, cat_features=[c for c in self.cat_cols if c in Xtr.columns])
                p = m.predict_proba(Xva)[:, 1]
                scores.append(average_precision_score(yva, p))

            return float(np.mean(scores)) if scores else 0.0

        return fn

    def train(self, train_df: pd.DataFrame):
        train_df = train_df.sort_values("scoring_date").reset_index(drop=True)
        X, y = self.preprocess(train_df)

        if y.nunique() < 2:
            raise ValueError("Training target has no variance.")

        split = int(len(X) * 0.8)
        Xtr, Xva = X.iloc[:split], X.iloc[split:]
        ytr, yva = y.iloc[:split], y.iloc[split:]

        logger.info("Running Optuna tuning...")
        study = optuna.create_study(direction="maximize")
        study.optimize(self._objective(Xtr, ytr), n_trials=8, show_progress_bar=False)

        best = study.best_params
        n_pos = ytr.sum()
        n_neg = len(ytr) - n_pos

        best.update({
            "loss_function": "Logloss",
            "eval_metric": "PRAUC",
            "scale_pos_weight": float(np.sqrt(n_neg / max(1, n_pos))),
            "random_seed": CFG.RANDOM_STATE,
            "verbose": False,
        })

        logger.info(f"Best params: {best}")

        self.model = CatBoostClassifier(**best)
        self.model.fit(Xtr, ytr, cat_features=[c for c in self.cat_cols if c in Xtr.columns])

        raw_va = self.model.predict_proba(Xva)[:, 1]
        self.calibrator.fit(raw_va, yva)
        p = self.calibrator.predict(raw_va)

        self.validation_predictions_ = p
        self.validation_truth_ = yva.values

        try:
            fi = self.model.get_feature_importance()
            self.feature_importance_ = pd.DataFrame({
                "feature": Xtr.columns,
                "importance": fi
            }).sort_values("importance", ascending=False).reset_index(drop=True)
        except Exception as e:
            logger.warning(f"Could not compute feature importance: {e}")
            self.feature_importance_ = pd.DataFrame(columns=["feature", "importance"])

        pred = (p >= 0.5).astype(int)

        logger.info(f"ROC-AUC : {roc_auc_score(yva, p):.4f}")
        logger.info(f"PR-AUC  : {average_precision_score(yva, p):.4f}")
        logger.info(f"Brier   : {brier_score_loss(yva, p):.4f}")
        logger.info("\n" + classification_report(yva, pred, zero_division=0))

    def predict_proba(self, df: pd.DataFrame) -> np.ndarray:
        X, _ = self.preprocess(df)
        raw = self.model.predict_proba(X)[:, 1]
        return self.calibrator.predict(raw)


# ============================================================
# NBA
# ============================================================

class StockPenalizer:
    @staticmethod
    def apply(base_probs: np.ndarray, stocks: pd.Series) -> np.ndarray:
        ratio = stocks.fillna(0).astype(float) / float(CFG.STOCK_THRESHOLD)
        return base_probs * np.minimum(1.0, np.maximum(0.0, ratio.values))


class NBAEngine:
    @staticmethod
    def pick_top_reason(row: pd.Series) -> str:
        checks = [
            ("high historical same-SKU conversion", row.get("same_sku_conversion_rate", 0) >= 0.50),
            ("strong overall historical conversion", row.get("historical_conversion_rate", 0) >= 0.40),
            ("recent engagement continuity", row.get("days_since_last_visit", 999) <= 14),
            ("high scheme sensitivity", row.get("scheme_sensitivity_index", 0) >= 0.40 and row.get("scheme_name_last_30d", "NO_SCHEME") != "NO_SCHEME"),
            ("healthy local stock availability", row.get("latest_closing_stock", 0) >= CFG.STOCK_THRESHOLD),
            ("dense nearby active pharmacy network", row.get("active_pharmacy_density", 0) >= 5),
            ("high patient load / demand capacity", row.get("patient_load", 0) > 0),
            ("strong lookalike cold-start similarity", row.get("cold_start_flag", 0) == 1 and row.get("cold_start_neighbor_conv_rate", 0) > 0.20),
            ("warning: elevated historical cancellations", row.get("cancellation_ratio", 0) > 0.30),
        ]
        for txt, cond in checks:
            if cond:
                return txt
        return "balanced multi-factor signal"

    @staticmethod
    def assign_dynamic_tiers(df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()

        non_zero = out[out["adjusted_propensity"] > 0].copy()
        out["recommendation_tier"] = "Tier 4 (Do Not Visit / Stockout)"

        if non_zero.empty:
            return out

        # Dynamic percentile bands on non-zero adjusted propensity
        p90 = non_zero["adjusted_propensity"].quantile(0.90)
        p60 = non_zero["adjusted_propensity"].quantile(0.60)
        p30 = non_zero["adjusted_propensity"].quantile(0.30)

        out.loc[out["adjusted_propensity"] > 0, "recommendation_tier"] = "Tier 3 (Low Priority)"
        out.loc[out["adjusted_propensity"] >= p30, "recommendation_tier"] = "Tier 3 (Low Priority)"
        out.loc[out["adjusted_propensity"] >= p60, "recommendation_tier"] = "Tier 2 (Medium Priority)"
        out.loc[out["adjusted_propensity"] >= p90, "recommendation_tier"] = "Tier 1 (High Priority)"

        return out

    @staticmethod
    def generate(scored_df: pd.DataFrame, model: DispenserModel) -> pd.DataFrame:
        out = scored_df.copy()
        out["base_propensity"] = model.predict_proba(out)
        out["adjusted_propensity"] = StockPenalizer.apply(out["base_propensity"], out["latest_closing_stock"]) * 100.0
        out["base_propensity"] = out["base_propensity"] * 100.0

        out = ExplorationPolicy.apply(out)
        out = NBAEngine.assign_dynamic_tiers(out)
        out["top_reason"] = out.apply(NBAEngine.pick_top_reason, axis=1)

        prompts = []
        for _, row in out.iterrows():
            if row["adjusted_propensity"] == 0:
                prompts.append(
                    f"DO NOT VISIT | Account: {row.get('account_name','Unknown')} | "
                    f"SKU: {row.get('sku_code','Unknown')} | Reason: local stock unavailable. Trigger replenishment first."
                )
                continue

            scheme_txt = (
                f"Use scheme '{row.get('scheme_name_last_30d','NO_SCHEME')}'."
                if row.get("scheme_name_last_30d", "NO_SCHEME") not in ["NO_SCHEME", "", None]
                else "No strong recent scheme signal; use standard commercial detailing."
            )
            mode_txt = (
                "Selected under exploration policy using cold-start similarity."
                if row.get("targeting_mode") == "Exploration"
                else "Selected under ranked exploitation policy."
            )
            cancel_txt = (
                f" Warning: cancellation ratio {row.get('cancellation_ratio',0):.1%}; check service/supply friction."
                if row.get("cancellation_ratio", 0) > 0.30 else ""
            )

            prompts.append(
                f"TARGET: {row.get('account_name','Unknown')} | Type: {row.get('account_type','Unknown')} | "
                f"Territory: {row.get('territory_name','Unknown')} | SKU: {row.get('sku_code','Unknown')} | "
                f"Base: {row.get('base_propensity',0):.1f}% | Adjusted: {row.get('adjusted_propensity',0):.1f}% | "
                f"Closing Stock: {row.get('latest_closing_stock',0):.0f} | Driver: {row.get('top_reason','balanced signal')}. "
                f"{scheme_txt} {mode_txt}{cancel_txt}"
            )

        out["next_best_action_prompt"] = prompts

        cols = [
            "account_id", "account_name", "account_type", "sku_code", "scoring_date",
            "territory_name", "pool_name", "mapped_distributor_id",
            "base_propensity", "adjusted_propensity", "recommendation_tier",
            "latest_closing_stock", "cancellation_ratio", "cold_start_flag",
            "targeting_mode", "top_reason", "next_best_action_prompt"
        ]
        cols = [c for c in cols if c in out.columns]
        return out[cols].sort_values("adjusted_propensity", ascending=False).reset_index(drop=True)


# ============================================================
# MASTER PIPELINE
# ============================================================

def run_pipeline():
    try:
        logger.info("=" * 80)
        logger.info("ROBUST FUTURE-PROOF PRIORITY UPGRADE PIPELINE STARTED")
        logger.info("=" * 80)

        accounts_df = AccountsParser().parse(DataUtils.stream_json_file(CFG.ACCOUNTS_PATH, CFG.STREAM_LIMIT))
        calls_h, calls_l = CallsParser().parse(DataUtils.stream_json_file(CFG.CALLS_PATH, CFG.STREAM_LIMIT))
        stock_df = StockParser().parse(DataUtils.stream_json_file(CFG.STOCK_PATH, CFG.STREAM_LIMIT))

        calls_df = EntityResolver.build_calls_table(calls_h, calls_l, accounts_df)
        fulfill_map = EntityResolver.build_fulfillment_mapping(calls_df)

        del calls_h, calls_l
        DataUtils.gc()

        start_date, end_date = CandidateGenerator.build_training_ranges(calls_df, stock_df)
        scoring_dates = CandidateGenerator.generate_scoring_dates(start_date, end_date)
        logger.info(f"Using {len(scoring_dates)} scoring dates: {scoring_dates.min()} -> {scoring_dates.max()}")

        snapshots = CandidateGenerator.build_dispenser_snapshots(accounts_df, calls_df, stock_df, scoring_dates)
        if snapshots.empty:
            raise ValueError("No snapshots generated.")

        labeled = TargetBuilderDispenser.build(snapshots, calls_df)
        del snapshots
        DataUtils.gc()

        fe = HistoricalFeatureBuilder(calls_df, accounts_df, stock_df)
        train_df = fe.run(labeled, fulfill_map)
        train_df = ColdStartFeatureBuilder.add(train_df)
        train_df = DataUtils.reduce_memory(train_df)

        del labeled
        DataUtils.gc()

        logger.info(f"Training dataframe shape: {train_df.shape}")
        logger.info(f"Training positive rate: {train_df['y_target'].mean():.4f}")

        model = DispenserModel()
        model.train(train_df)

        # Diagnostics on validation
        eval_topk = ModelDiagnostics.top_k_metrics(
            pd.Series(model.validation_truth_),
            model.validation_predictions_,
            k_list=[0.01, 0.05, 0.10, 0.20]
        )
        eval_deciles = ModelDiagnostics.decile_gains_table(
            pd.Series(model.validation_truth_),
            model.validation_predictions_,
            n_bins=10
        )
        calibration_df = ModelDiagnostics.calibration_table(
            pd.Series(model.validation_truth_),
            model.validation_predictions_,
            n_bins=10
        )
        score_dist_df = ModelDiagnostics.score_distribution(model.validation_predictions_)

        logger.info("Top-k metrics:")
        logger.info("\n" + eval_topk.to_string(index=False))

        logger.info("Decile gains table:")
        logger.info("\n" + eval_deciles.to_string(index=False))

        logger.info("Calibration table:")
        logger.info("\n" + calibration_df.to_string(index=False))

        logger.info("Score distribution:")
        logger.info("\n" + score_dist_df.to_string(index=False))

        # Latest scoring
        latest_date = scoring_dates.max()
        latest_snap = CandidateGenerator.build_dispenser_snapshots(
            accounts_df,
            calls_df,
            stock_df,
            pd.DatetimeIndex([latest_date])
        )

        latest_df = fe.run(latest_snap, fulfill_map)
        latest_df = ColdStartFeatureBuilder.add(latest_df)
        latest_df = DataUtils.reduce_memory(latest_df)

        mapping_quality_df = ModelDiagnostics.mapping_quality(latest_df)
        logger.info("Mapping quality diagnostics:")
        logger.info("\n" + mapping_quality_df.to_string(index=False))

        nba_df = NBAEngine.generate(latest_df, model)

        # Save outputs
        nba_df.to_csv(CFG.OUTPUT_FILENAME, index=False)
        logger.info(f"Saved scoring output: {CFG.OUTPUT_FILENAME}")

        eval_df = pd.concat(
            [
                eval_topk.assign(section="top_k"),
                eval_deciles.assign(section="deciles"),
                score_dist_df.assign(section="score_distribution"),
                mapping_quality_df.assign(section="mapping_quality"),
            ],
            ignore_index=True,
            sort=False
        )
        eval_df.to_csv(CFG.EVAL_OUTPUT_FILENAME, index=False)
        logger.info(f"Saved evaluation output: {CFG.EVAL_OUTPUT_FILENAME}")

        calibration_df.to_csv(CFG.CALIBRATION_FILENAME, index=False)
        logger.info(f"Saved calibration diagnostics: {CFG.CALIBRATION_FILENAME}")

        if model.feature_importance_ is not None:
            model.feature_importance_.to_csv(CFG.FEATURE_IMPORTANCE_FILENAME, index=False)
            logger.info(f"Saved feature importance: {CFG.FEATURE_IMPORTANCE_FILENAME}")

        # Upload to S3
        try:
            s3 = boto3.client("s3")
            for fname in [
                CFG.OUTPUT_FILENAME,
                CFG.EVAL_OUTPUT_FILENAME,
                CFG.CALIBRATION_FILENAME,
                CFG.FEATURE_IMPORTANCE_FILENAME
            ]:
                try:
                    s3.upload_file(fname, CFG.S3_BUCKET, f"outputs/{fname}")
                    logger.info(f"Uploaded to s3://{CFG.S3_BUCKET}/outputs/{fname}")
                except Exception as inner_upload_err:
                    logger.warning(f"Upload failed for {fname}: {inner_upload_err}")
        except Exception as upload_err:
            logger.warning(f"S3 client/upload setup failed: {upload_err}")

        logger.info("Top 10 recommendations:")
        logger.info("\n" + nba_df.head(10).to_string(index=False))

        if model.feature_importance_ is not None and not model.feature_importance_.empty:
            logger.info("Top 20 feature importances:")
            logger.info("\n" + model.feature_importance_.head(20).to_string(index=False))

        logger.info("=" * 80)
        logger.info("ROBUST FUTURE-PROOF PRIORITY UPGRADE PIPELINE COMPLETED")
        logger.info("=" * 80)

        return {
            "accounts_df": accounts_df,
            "calls_df": calls_df,
            "stock_df": stock_df,
            "train_df": train_df,
            "latest_df": latest_df,
            "nba_df": nba_df,
            "model": model,
            "eval_topk": eval_topk,
            "eval_deciles": eval_deciles,
            "calibration_df": calibration_df,
            "score_dist_df": score_dist_df,
            "mapping_quality_df": mapping_quality_df,
            "feature_importance_df": model.feature_importance_,
        }

    except Exception as e:
        logger.error(f"Pipeline failed: {e}")
        logger.error(traceback.format_exc())
        raise


if __name__ == "__main__":
    artifacts = run_pipeline()